In [1]:
import json
import re

def clean_answer(answer):
    answer_clean = re.sub('[^\d\.]', '', answer)
    if answer_clean and answer_clean[-1] == '.':
        answer_clean = answer_clean[:-1]
    if '.' in answer_clean:
        if answer_clean[-1] == '0' and answer_clean[-2] == '0':
            answer_clean = answer_clean.rstrip('0')
            answer_clean = answer_clean[:-1]
        if answer_clean and answer_clean[-1] == '.':
            answer_clean = answer_clean[:-1]
    return answer_clean

def get_true_answer(correct_answer):
    match = re.search(r'####\s*(\d+)', correct_answer)
    if match:
        return match.group(1)
    else:
        return 'NA'

def tally_and_collect_answers(data):
    correct_count = 0
    incorrect_count = 0
    results_list = []
    
    for entry in data:
        model_output = clean_answer(entry["model_response"]["output"])
        true_answer = get_true_answer(entry["correct_answer"])
        is_correct = model_output == true_answer
        
        if is_correct:
            correct_count += 1
        else:
            incorrect_count += 1
        
        results_list.append({
            "question": entry["question"],
            "model_output": model_output,
            "true_answer": true_answer,
            "is_correct": is_correct
        })
    
    return correct_count, incorrect_count, results_list

# Load the JSON file with your dataset
with open("/p/llmreliability/test_repos/ReAct/SoK_Experiments/results/GSM8k/llama3-groq-8b-8192-tool-use-preview_gsm8k_react_results_multiple_choice.json", "r") as file:
    data = json.load(file)

# Tally the correct and incorrect answers and collect the results
correct, incorrect, results_list = tally_and_collect_answers(data)

print(f"Correct Answers: {correct}")
print(f"Incorrect Answers: {incorrect}")

# Print out the detailed results
for result in results_list:
    print("\nQuestion:", result["question"])
    print("Model Output:", result["model_output"])
    print("True Answer:", result["true_answer"])
    print("Correct:", result["is_correct"])



TypeError: string indices must be integers

In [3]:
def calculate_accuracy(data):
    correct_count = 0
    total_count = len(data)
    answered_count = 0

    for item in data:
        model_response = item['model_response']
        correct_answer = item['correct_answer']
        
        if model_response == "Agent stopped due to iteration limit or time limit.":
            continue  # Skip this item as the model didn't provide an answer
        
        answered_count += 1
        
        # Check if the model response is a letter (A, B, C, D)
        if model_response in ['A', 'B', 'C', 'D']:
            model_value = item['options'][model_response]
        # Check if the model response is in the format "A) value"
        elif model_response.startswith(('A)', 'B)', 'C)', 'D)')):
            letter = model_response[0]
            model_value = item['options'][letter]
        else:
            continue  # Skip this item if the model's response is not in the expected format
        
        if abs(model_value - correct_answer) < 0.01:  # Use a small tolerance for floating-point comparison
            correct_count += 1

    accuracy = (correct_count / answered_count) * 100 if answered_count > 0 else 0
    return accuracy, answered_count, total_count

def load_data_from_json(file_path):
    with open(file_path, 'r') as file:
        data = json.load(file)
    return data

# Example usage
file_path = '/p/llmreliability/test_repos/ReAct/SoK_Experiments/results/GSM8k/llama3-groq-8b-8192-tool-use-preview_gsm8k_react_results_multiple_choice.json'  # Replace with the actual path to your JSON file
data = load_data_from_json(file_path)
accuracy, answered_count, total_count = calculate_accuracy(data)
print(f"Model accuracy: {accuracy:.2f}%")
print(f"Questions answered: {answered_count}/{total_count}")

       

Model accuracy: 48.39%
Questions answered: 93/99
